# Week 5, Day 4: Least squares: fitting linear encoding models

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w005/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Fit a line three ways

A neuron's rate grows roughly linearly with contrast. Build the design matrix `X` (a column of ones, then the contrasts), solve the normal equations with `np.linalg.solve` to get `w_normal`, and compare with `np.linalg.lstsq` (`w_lstsq`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
c = np.repeat([0.05, 0.1, 0.2, 0.4, 0.8], 20)                 # contrasts, 20 trials each
y = 5 + 30 * c + rng.normal(0, 3, c.size)                     # rate (Hz)

X = ...
w_normal = ...
w_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
print("baseline and slope:", np.round(w_normal, 2), np.round(w_lstsq, 2))

plt.figure(figsize=(5, 3)); plt.plot(c, y, "o", alpha=0.4)
cc = np.linspace(0, 0.85, 50); plt.plot(cc, w_normal[0] + w_normal[1] * cc)
plt.xlabel("contrast"); plt.ylabel("rate (Hz)"); plt.show()

In [ ]:
import numpy as np
def test_design():
    "X has a column of ones and a column of contrasts"
    assert X.shape == (len(c), 2) and np.allclose(X[:, 0], 1) and np.allclose(X[:, 1], c)

def test_normal():
    "w_normal solves the normal equations"
    assert np.allclose(X.T @ X @ w_normal, X.T @ y)

def test_agree():
    "normal equations, lstsq and polyfit agree"
    assert np.allclose(w_normal, w_lstsq)
    assert np.allclose(w_normal[::-1], np.polyfit(c, y, 1))

test_design()
test_normal()
test_agree()
print("All checks passed")

### Exercise: A tuning curve by linear regression

Fit the three-weight model to `rates` measured at orientations `theta` (radians). Store the weights in `w`, the preferred orientation in degrees (0 to 180) in `pref_deg`, and the R² in `r2`.

In [ ]:
theta = np.radians(np.repeat(np.arange(0, 180, 15), 10))      # 12 orientations, 10 trials each
rates = 10 + 8 * np.cos(2 * (theta - np.radians(65))) + rng.normal(0, 2, theta.size)

Xd = ...
w, *_ = ...
pref_deg = ...
r2 = ...
print("weights:", np.round(w, 2), " preferred:", round(pref_deg, 1), "deg  R2:", round(r2, 3))

In [ ]:
import numpy as np
def test_fit():
    "w has three weights and fits the data by least squares"
    assert w.shape == (3,)
    Xd = np.column_stack([np.ones_like(theta), np.cos(2 * theta), np.sin(2 * theta)])
    assert np.allclose(Xd.T @ (rates - Xd @ w), 0, atol=1e-6)

def test_pref():
    "pref_deg is within 8 degrees of the true preference"
    assert abs(((pref_deg - 65) + 90) % 180 - 90) < 8

def test_r2():
    "r2 is the coefficient of determination of the fit"
    Xd = np.column_stack([np.ones_like(theta), np.cos(2 * theta), np.sin(2 * theta)])
    pred = Xd @ w
    assert abs(r2 - (1 - np.sum((rates - pred) ** 2) / np.sum((rates - rates.mean()) ** 2))) < 1e-9

test_fit()
test_pref()
test_r2()
print("All checks passed")

### Exercise: White noise against correlated stimuli

`Xc` holds 2,000 correlated (smoothed) stimuli and `yc` the responses. Compute `sta_corr` (the STA) and `ls_corr` (least squares). Plot both with the true receptive field `w_true`. Which matches?

In [ ]:
x = np.arange(40)
w_true = np.exp(-(x - 20) ** 2 / 30) * np.cos((x - 20) / 1.2)              # a 1D Gabor-like receptive field
white = rng.normal(size=(2000, 40))
kernel = np.exp(-np.arange(-6, 7) ** 2 / 8)
Xc = 0.5 * white + 9 * np.array([np.convolve(s, kernel, mode="same") for s in white]) / kernel.sum()   # neighbouring pixels correlated
yc = Xc @ w_true + rng.normal(0, 2, 2000)

sta_corr = ...
ls_corr = ...
plt.figure(figsize=(6, 3))
plt.plot(w_true / np.abs(w_true).max(), "k", lw=3, label="true")
plt.plot(sta_corr / np.abs(sta_corr).max(), label="spike-triggered average")
plt.plot(ls_corr / np.abs(ls_corr).max(), label="least squares")
plt.xlabel("pixel"); plt.ylabel("weight (scaled)"); plt.legend(frameon=False, fontsize=8); plt.show()
print("correlation with truth: STA", round(np.corrcoef(sta_corr, w_true)[0, 1], 3), " LS", round(np.corrcoef(ls_corr, w_true)[0, 1], 3))

In [ ]:
import numpy as np
def test_estimates():
    "sta and ls have the right shapes and formulas"
    assert sta_corr.shape == (40,) and ls_corr.shape == (40,)
    assert np.allclose(sta_corr, Xc.T @ yc / len(yc))
    assert np.allclose(ls_corr, np.linalg.lstsq(Xc, yc, rcond=None)[0])

def test_ls_better():
    "least squares recovers the true receptive field better than the STA for correlated stimuli"
    r_sta = np.corrcoef(sta_corr, w_true)[0, 1]; r_ls = np.corrcoef(ls_corr, w_true)[0, 1]
    assert r_ls > 0.9 and r_ls > r_sta + 0.05, f"STA {r_sta:.2f}, LS {r_ls:.2f}"

test_estimates()
test_ls_better()
print("All checks passed")